In [0]:
import json
import ast
from pydantic import BaseModel, field_validator
from typing import List, Optional, ClassVar
from decimal import Decimal
import pandas as pd


class FacilityRecord(BaseModel):

    model_config = {"arbitrary_types_allowed": True}

    name: str
    facility_type: Optional[str] = None
    operator_type: Optional[str] = None

    phone_numbers: Optional[str] = None
    officialPhone: Optional[str] = None
    email: Optional[str] = None
    websites: Optional[str] = None
    officialWebsite: Optional[str] = None

    city: Optional[str] = None
    state: Optional[str] = None
    pincode: Optional[str] = None
    lat: Optional[float] = None
    lon: Optional[float] = None

    specialties: List[str] = []
    procedures: List[str] = []
    equipment: List[str] = []
    capabilities: List[str] = []

    number_doctors: Optional[int] = None
    year_established: Optional[int] = None
    n_followers: Optional[int] = None

    raw_notes: Optional[str] = None

    trust_score: Optional[float] = None
    trust_flags: List[str] = []
    reasoning: Optional[str] = None
    ai_reviewed: bool = False


    #Validators

    EMPTY_VALUES: ClassVar[tuple] = (
        "null", "nan", "none", "", "unknown",
        "n/a", "na", "-", "not available"
    )

    @field_validator("specialties", "procedures", "equipment", "capabilities", mode="before")
    @classmethod
    def parse_json_list(cls, v):
        if v is None:
            return []
        if isinstance(v, list):
            return v
        s = str(v).strip()
        if s.lower() in cls.EMPTY_VALUES or s == "[]":
            return []
        try:
            parsed = json.loads(s)
            return parsed if isinstance(parsed, list) else [str(parsed)]
        except Exception:
            try:
                parsed = ast.literal_eval(s)
                return parsed if isinstance(parsed, list) else [str(parsed)]
            except Exception:
                return [s]

    @field_validator("lat", "lon", mode="before")
    @classmethod
    def parse_decimal(cls, v):
        if v is None:
            return None
        try:
            return float(v)
        except Exception:
            return None

    @field_validator(
        "facility_type", "operator_type", "city", "state",
        "pincode", "email", "officialWebsite", "officialPhone",
        "websites", "phone_numbers",
        mode="before"
    )
    @classmethod
    def clean_null_strings(cls, v):
        if v is None or str(v).strip().lower() in cls.EMPTY_VALUES:
            return None
        return str(v).strip()

    @field_validator("pincode", mode="after")
    @classmethod
    def validate_pincode(cls, v):
        if v:
            clean = str(v).replace(".0", "").strip()
            return clean if (len(clean) == 6 and clean.isdigit()) else None
        return v

    @field_validator("number_doctors", "year_established", "n_followers", mode="before")
    @classmethod
    def parse_nullable_int(cls, v):
        if v is None or str(v).strip().lower() in cls.EMPTY_VALUES:
            return None
        try:
            return int(float(str(v)))
        except Exception:
            return None

    @field_validator("trust_score", mode="after")
    @classmethod
    def score_range(cls, v):
        if v is not None and not (0.0 <= v <= 1.0):
            raise ValueError("trust_score muss zwischen 0.0 und 1.0 liegen")
        return v

    @classmethod
    def from_row(cls, row: dict) -> "FacilityRecord":

        desc = str(row.get("description") or "")
        cap  = str(row.get("capability") or "")
        proc = str(row.get("procedure") or "")

        notes_parts = [
            p for p in [desc, cap, proc]
            if p.strip().lower() not in cls.EMPTY_VALUES
        ]
        combined_notes = " | ".join(notes_parts)

        return cls(
            name             = str(row.get("name", "Unbekannt")),
            facility_type    = row.get("facilityTypeId"),
            operator_type    = row.get("operatorTypeId"),
            phone_numbers    = row.get("phone_numbers"),
            officialPhone    = row.get("officialPhone"),
            email            = row.get("email"),
            websites         = row.get("websites"),
            officialWebsite  = row.get("officialWebsite"),
            city             = row.get("address_city"),
            state            = row.get("address_stateOrRegion"),
            pincode          = str(row.get("address_zipOrPostcode", "")),
            lat              = row.get("latitude"),
            lon              = row.get("longitude"),
            specialties      = row.get("specialties"),
            procedures       = row.get("procedure"),
            equipment        = row.get("equipment"),
            capabilities     = row.get("capability"),
            number_doctors   = row.get("numberDoctors"),
            year_established = row.get("yearEstablished"),
            n_followers      = row.get("engagement_metrics_n_followers"),
            raw_notes        = combined_notes if combined_notes else None,
        )    



from typing import Tuple, List

def _text(*fields) -> str:
    parts = []
    for f in fields:
        if isinstance(f, list):
            parts.extend([str(x).lower() for x in f if x])
        elif f:
            parts.append(str(f).lower())
    return " ".join(parts)

def _has(text: str, *keywords) -> bool:
    return any(kw in text for kw in keywords)

def compute_trust_score(record: FacilityRecord) -> Tuple[float, List[str]]:
    flags = []
    score = 1.0

    all_text = _text(
        record.raw_notes,
        record.specialties,
        record.procedures,
        record.capabilities
    )
    staff_text   = _text(record.staff) if hasattr(record, 'staff') else ""
    equip_text   = _text(record.equipment)
    special_text = _text(record.specialties)

    if _has(all_text, "surgery", "surgical", "appendectomy", "operation", "laparoscop", "orthopedic", "transplant", "cardiac", "neurosurgery", "plastic surgery") \
    and not _has(all_text + equip_text, "anesthes", "anaesthes", "anesthesia machine", "anesthetist", "anesthesiology"):
        flags.append("⚠️ Surgery claimed, but no anesthetist found")
        score -= 0.30

    if _has(all_text, "icu", "intensive care", "critical care", "neonatal icu", "nicu", "picu") \
    and not _has(equip_text + all_text, "ventilator", "monitor", "oxygen", "defibrillator", "life support", "infusion pump", "suction", "ecg", "pulse oximeter"):
        flags.append("⚠️ ICU claimed, but no ventilators listed")
        score -= 0.25

    if _has(all_text, "24/7", "24 hour", "always open", "round the clock", "emergency services", "ambulance") \
    and _has(all_text, "part-time", "part time", "parttime", "visiting doctor", "visiting consultant", "limited hours"):
        flags.append("⚠️ 24/7 claimed, but part-time staff mentioned")
        score -= 0.20

    if _has(_text(record.name, record.raw_notes), "advanced", "multispecial", "super special", "state-of-the-art", "modern", "comprehensive") \
    and len(record.specialties) < 2:
        flags.append("⚠️ 'Advanced' claimed, but few specialties listed")
        score -= 0.20

    if len(record.specialties) > 3 and len(record.raw_notes or "") < 50:
        flags.append("⚠️ Many specialties, but little description text")
        score -= 0.15

    if _has(all_text, "emergency", "trauma", "notfall", "casualty", "urgent care") \
    and not _has(all_text, "24/7", "24 hour", "always open", "ambulance"):
        flags.append("⚠️ Emergency mentioned, but no 24/7 indication")
        score -= 0.10

    if not record.lat or not record.lon:
        flags.append("ℹ️ No GPS coordinates — geographic search limited")
        score -= 0.05

    if not record.websites and not record.officialWebsite:
        flags.append("ℹ️ No website listed")
        score -= 0.05

    if not record.email:
        flags.append("ℹ️ No email listed")
        score -= 0.05

    if not record.phone_numbers and not record.officialPhone:
        flags.append("ℹ️ No phone number listed")
        score -= 0.05

    return round(max(score, 0.0), 2), flags


import json

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

def ai_trust_review(record: FacilityRecord) -> dict:
    prompt = f"""You are a medical quality auditor reviewing Indian healthcare facilities.

    FACILITY: {record.name}
    TYPE: {record.facility_type} | STATE: {record.state} | CITY: {record.city}

    DESCRIPTION:
    {record.raw_notes or "No description available"}

    SPECIALTIES: {', '.join(record.specialties) if record.specialties else "None listed"}
    EQUIPMENT:   {', '.join(record.equipment) if record.equipment else "None listed"}
    PROCEDURES:  {', '.join(record.procedures) if record.procedures else "None listed"}

    AUTOMATICALLY DETECTED CONTRADICTIONS:
    {chr(10).join(record.trust_flags) if record.trust_flags else "None"}

    YOUR TASK:
    1. Read the full description carefully
    2. Check if the detected contradictions are justified, or if the text actually explains them
    3. Find any additional contradictions the rules missed
    4. Assign a final trust score from 0.0 (very suspicious) to 1.0 (fully trustworthy)

    Respond ONLY in this exact JSON format, nothing else:
    {{
    "trust_score": 0.85,
    "confirmed_flags": ["flags that are genuinely problematic"],
    "dismissed_flags": ["flags that the text actually explains"],
    "new_flags": ["new contradictions you found"],
    "reasoning": "One sentence explaining the score"
    }}"""

    response = w.serving_endpoints.query(
        name="databricks-meta-llama-3-3-70b-instruct",
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = response.choices[0].message.content
    
    start = raw.find("{")
    end   = raw.rfind("}") + 1
    return json.loads(raw[start:end])


def full_trust_pipeline(record: FacilityRecord) -> FacilityRecord:
   
    records = [FacilityRecord.from_row(row) for _, row in df.iterrows()]
    for record in records:
        score, flags = compute_trust_score(record)
        record.trust_score = score
        record.trust_flags = flags

    needs_review = record.trust_score < 0.8 or len(record.trust_flags) > 0
    if needs_review:
        try:
            ai_result = ai_trust_review(record)
            record.trust_score = ai_result["trust_score"]
            record.trust_flags = (
                ai_result.get("confirmed_flags", []) +
                ai_result.get("new_flags", [])
            )
            record.reasoning = ai_result["reasoning"]
            record.ai_reviewed = True
            
        except Exception as e:
            record.reasoning = f"AI review failed: {e} — rule-based score kept"
            record.ai_reviewed = False
    else:
        record.reasoning = "No contradictions found — automatically verified"
        record.ai_reviewed = False
    return record




import time

# Recors creation
rows = spark.table("workspace.default.base_table").collect()
records = [FacilityRecord.from_row(row.asDict()) for row in rows]
print(f"{len(records)} Records erstellt")

# Basic trust scores
for record in records:
    score, flags = compute_trust_score(record)
    record.trust_score = score
    record.trust_flags = flags

needs_review = [r for r in records if r.trust_score < 0.8 or len(r.trust_flags) > 0]
print(f"Automatisch verifiziert: {len(records) - len(needs_review)}")
print(f"Brauchen KI-Review: {len(needs_review)}")

# KI-Review
for i, record in enumerate(needs_review):
    try:
        ai_result = ai_trust_review(record)

        record.trust_score = ai_result["trust_score"]
        record.trust_flags = (
            ai_result.get("confirmed_flags", []) +
            ai_result.get("new_flags", [])
        )
        record.reasoning   = ai_result["reasoning"]
        record.ai_reviewed = True

    except Exception as e:
        record.reasoning   = f"AI review failed: {e} — rule-based score kept"
        record.ai_reviewed = False

    # Progress
    if i % 100 == 0:
        print(f"   {i}/{len(needs_review)} reviewed...")

    time.sleep(0.1)

print(f"KI-Review abgeschlossen")
print(f"Erfolgreich: {sum(1 for r in needs_review if r.ai_reviewed)}")
print(f"Fehlgeschlagen: {sum(1 for r in needs_review if not r.ai_reviewed)}")

# Storing as DataFrame
print("\nErgebnis speichern...")
df_scored = pd.DataFrame([r.model_dump() for r in records])
df_scored.to_csv("../data/facilities_scored.csv", index=False)